In [1]:
import pandas as pd

In [2]:

# Load the datasets
pm = pd.read_csv("../Data/2026-10_pms7003.csv", sep=";")
weather = pd.read_csv("../Data/2026-10_bmp280.csv", sep=";")

C:\Users\ABDQUDUS\AppData\Local\Temp\ipykernel_21092\3077418570.py:2: DtypeWarning: Columns (0: P1, 1: P2, 2: P0) have mixed types. Specify dtype option on import or set low_memory=False.
  pm = pd.read_csv("../Data/2026-10_pms7003.csv", sep=";")
C:\Users\ABDQUDUS\AppData\Local\Temp\ipykernel_21092\3077418570.py:3: DtypeWarning: Columns (0: temperature) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.read_csv("../Data/2026-10_bmp280.csv", sep=";")


In [3]:
# Convert timestamps
pm["timestamp"] = pd.to_datetime(pm["timestamp"])
weather["timestamp"] = pd.to_datetime(weather["timestamp"])


In [4]:
# Normalize location IDs
pm["location"] = pm["location"].astype(str)
weather["location"] = weather["location"].astype(str)


In [5]:
# Check whether the datasets share location IDs
common_locations = (
    set(pm["location"]) & set(weather["location"])
)

print("Common locations:", len(common_locations))


Common locations: 8


In [6]:
# Create hourly timestamps
pm["hour"] = pm["timestamp"].dt.floor("h")
weather["hour"] = weather["timestamp"].dt.floor("h")


In [10]:
pm.head()

,sensor_id,sensor_type,location,lat,lon,timestamp,P1,P2,P0,hour
0,92442,PMS7003,85294,49.818813,23.985120,2026-10-01 00:00:01,11.0,10.0,9.0,2026-10-01
1,57273,PMS7003,43121,52.264400,20.643095,2026-10-01 00:00:01,14.0,10.0,NaN,2026-10-01
2,97299,PMS7003,90057,45.844000,20.458000,2026-10-01 00:00:01,47.5,35.5,24.5,2026-10-01
3,90109,PMS7003,82080,33.682000,72.978000,2026-10-01 00:00:01,53.0,42.4,29.4,2026-10-01
4,57623,PMS7003,43448,52.264000,20.644000,2026-10-01 00:00:01,9.0,6.0,NaN,2026-10-01


In [9]:
pm['P1'].astype(float)
pm['P2'].astype(float)
pm['lat'].astype(float)
pm['lon'].astype(float)

ValueError: could not convert string to float: 'unavailable'

In [7]:
# Aggregate pollution measurements
pm_hourly = pm.groupby(
    ["location", "hour"], as_index=False
).agg(
    PM10=("P1", "mean"),
    PM25=("P2", "mean"),
    latitude=("lat", "mean"),
    longitude=("lon", "mean")
)


TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
# Aggregate meteorological measurements
weather_hourly = weather.groupby(
    ["location", "hour"], as_index=False
).agg(
    temperature=("temperature", "mean"),
    humidity=("humidity", "mean"),
    pressure=("pressure", "mean")
)


In [ ]:
# Join both datasets
final_df = pd.merge(
    pm_hourly,
    weather_hourly,
    on=["location", "hour"],
    how="inner"
)


In [ ]:
# Save the resulting dataset
final_df.to_csv("morocco_air_quality_dataset.csv", index=False)

print(final_df.head())
print("Final dataset shape:", final_df.shape)